# Milestone 6 Lab, notebook 3 of 8: The review desk

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| **3. The review desk** (you are here) | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **0**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

Since milestone 3 some findings have ended in **needs review**. The record stays in the figures, the answer carries a warning, and then nothing happens. Nobody can act on it.

In this notebook you build the missing piece: a **review desk**. You decide each finding, your decision is recorded with your name, and the next run obeys it.

No AI is used. Deciding a finding is a person's job, and applying the decision is code's.

> Run notebook 1 first.

In [ ]:

import pandas as pd
from lab_helpers import LAB_DECISIONS, LAB_E2E, show_doc

from retail_profit_stack.e2e import build_close

workspace = build_close(LAB_E2E)
if LAB_DECISIONS.exists():
    LAB_DECISIONS.unlink()

The first cell rebuilt the system to month-end close and cleared any decisions from an earlier run of this notebook.

## Step 1: The design

In [ ]:
show_doc("HLD.md", "6. The review desk")

## Step 2: What is waiting

Run the pipeline once, giving it the file where decisions will be kept. The file does not exist yet, so nothing is decided.

In [ ]:
from retail_profit_stack.transform.pipeline import run_pipeline

run = run_pipeline("sql", source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                   note="Before any decision", decisions_file=LAB_DECISIONS)

run.findings[["finding_id", "check_name", "month", "product_id", "outcome", "severity"]]

Four findings need review. `open_reviews` lists the ones a person can decide: those that point at a record.

In [ ]:
from retail_profit_stack.quality.review import open_reviews

waiting = open_reviews(run.findings, run.finding_records)

for item in waiting:
    print(item["record_key"])
    print("   ", item["detail"])
    print("   ", item["record"])

Three records. The fourth finding, the camp stove's missing cost, is not on the desk: there is no record to decide. Only the cost itself can fix it, and in notebook 2 it arrived.

Each record has a **key**, such as `refund_exceeds_line|refunds|REF-0002`: the check, the table, and the record's own id. We do not use the finding's number, because finding numbers are given in order on each run and can change. You saw that in notebook 2, where the list of findings shifted.

## Step 3: Decide

Look at each record as a data engineer would.

**The $60 refund on a $25 sun hat line.** Nobody refunds more than was paid. This was typed wrongly. **Exclude** it.

**The shipment dated one day before its order.** Orders are sometimes keyed in the morning after. The shipment is real. **Accept** it.

**The refund dated twelve days before its order.** That needs a call to customer service. Leave it undecided for now.

Put your own name in.

In [ ]:
from retail_profit_stack.quality.review import decide

ME = "your name"

decide(LAB_DECISIONS, "refund_exceeds_line|refunds|REF-0002", "exclude", ME,
       note="No refund can be larger than its line. Typed wrongly.")

shipment_key = next(item["record_key"] for item in waiting if "|shipments|" in item["record_key"])
decide(LAB_DECISIONS, shipment_key, "accept", ME, note="The order was recorded a day late.")

print(LAB_DECISIONS.read_text(encoding="utf-8"))

That file is the whole review desk: each decision, who made it, why, and when. Nothing has changed in the data yet.

## Step 4: The next run obeys

In [ ]:
after = run_pipeline("sql", source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                     note="After two decisions", decisions_file=LAB_DECISIONS, incremental=True)

print("Mode:", after.record["mode"])
after.findings[["finding_id", "check_name", "product_id", "outcome", "severity"]]

Two findings changed. The excluded refund is now **quarantine**, like a duplicate. The accepted shipment is **reviewed**, and its severity is low. The undecided refund still needs review.

Notice the mode: **full**, although we asked for an incremental run. A new decision can move a record in or out of the figures, so the pipeline checks everything again. It decided that itself.

Here is what the reviewed finding now says.

In [ ]:
for description in after.findings[after.findings["outcome"] != "needs_review"]["description"].tail(2):
    print("-", description)

## Step 5: What moved

Excluding the refund must change the sun hat's September net sales by exactly the $60. Accepting the shipment must change nothing.

In [ ]:
from retail_profit_stack.lineage.compare import compare_runs

comparison = compare_runs(run.run_id, after.run_id, workspace.warehouse)

figures = pd.DataFrame(comparison["figures_changed"])
figures[figures["metric"].isin(["refunds", "net_sales", "contribution_profit"])]

In [ ]:
print("Rows set aside:", after.quarantined_counts)
print("Lineage still adds up:", after.record["lineage_adds_up"])

The refund is set aside with the duplicates, not deleted. And the lineage still adds up, because an excluded record leaves the lineage too.

## Step 6: Confidence follows the decision

At close, a question about the sun hat's September figure got **medium** confidence, because of that refund. Ask the quality tool now.

In [ ]:
from retail_profit_stack import tools
from retail_profit_stack.store import Store


def store_for(run_id):
    return Store(source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                 run_id=run_id, state_file=workspace.places.state_file)


before_decision = tools.get_quality_flags("2026-09", "sun hat", store=store_for(run.run_id))
after_decision = tools.get_quality_flags("2026-09", "sun hat", store=store_for(after.run_id))

print("Before the decision:", before_decision.confidence_level)
print("After the decision: ", after_decision.confidence_level)

The problem was dealt with by a person, so it no longer lowers confidence. The finding is still listed, so the leader can see what was done.

## Step 7: A decision can be taken back

In [ ]:
from retail_profit_stack.quality.review import read_decisions, remove_decision

remove_decision(LAB_DECISIONS, shipment_key)

print(list(read_decisions(LAB_DECISIONS)))

On the next run that shipment would need review again.

## Step 8: A file that never came

One more warning had nowhere to go. In milestone 5 a refunds file was rejected, and a complete file replaced it. What if the complete file had never arrived? The figures would be missing records, and nobody would be told.

Build a small copy of the landing folder without the complete file, and ask the quality tool.

In [ ]:
import shutil

from lab_helpers import NOTEBOOKS

from retail_profit_stack.ingest.dispatch import load_batch
from retail_profit_stack.ingest.feeds import Places

folder = NOTEBOOKS / "lab_e2e" / "missing_file"
shutil.copytree(workspace.places.landing, folder / "landing")
(folder / "landing" / "refunds_2026-10-12_complete.csv").unlink()

places = Places(folder / "landing", folder / "loaded", folder / "state.sqlite", workspace.places.mappings_file)
load_batch(places=places)
run_pipeline("sql", source_dir=places.loaded, warehouse_dir=folder / "warehouse")

store = Store(source_dir=places.loaded, warehouse_dir=folder / "warehouse", state_file=places.state_file)
result = tools.get_quality_flags("2026-10", store=store)

print("Files missing:", result.files_missing)
print("Confidence:   ", result.confidence_level)
print(result.confidence_reasons[-1])

A rejected file that no loaded file has replaced is reported for its month, and confidence is low. The analyst's instructions tell it to name the file in its answer.

## What we now know

- A **review desk** is a file of decisions: the record, the decision, who, why, and when.
- **Exclude** sets a record aside, like a duplicate. **Accept** keeps it and marks the finding reviewed.
- A decision names a **record**, not a finding number, so it still applies when the list of findings changes.
- Code applies the decision on the next run. Confidence follows, and the lineage still adds up.
- A decision can be taken back. A rejected file that was never replaced is now a warning.

**Next:** notebook 4. The pipeline now does several things in one run. We look at a run's health, and ask the analyst to explain a restatement.